In [13]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load dataset
df = pd.read_csv('/content/imdb_top_1000.csv')

# Filter only entries with a title (which are primarily movies in this dataset)
df_movies = df[df['Series_Title'].notnull()]
df_movies = df_movies[df_movies['Certificate'].notnull()]  # optional cleanup

# Use Genre + Overview for content
df_movies['content'] = df_movies['Genre'] + " " + df_movies['Overview']

# TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['content'])

# Cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Index mapping
indices = pd.Series(df_movies.index, index=df_movies['Series_Title']).drop_duplicates()

# Recommendation function for movies
def recommend_movie(title, top_n=10):
    if title not in indices:
        print(f"Error: The movie title '{title}' was not found in the dataset.")
        print("Available titles start with:")
        print(df_movies['Series_Title'].head().tolist())
        return pd.Series([]) # Return an empty series or a message

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]

    movie_indices = [i[0] for i in sim_scores]
    return df_movies['Series_Title'].iloc[movie_indices]

# Example: Recommend 10 similar movies to 'The Shawshank Redemption'
print(recommend_movie("Interstellar", 10))

654                   Gattaca
357              The Avengers
745                   Gravity
343              The Revenant
106                    Aliens
677                  Predator
60     Avengers: Infinity War
275              Blade Runner
329               The Martian
842                 Airplane!
Name: Series_Title, dtype: object
